# Agentic RAG Challenge -- Student Academic Intelligence System

---

## Objective

Build an **Agentic RAG pipeline** that answers 50 test queries about student performance by reasoning over structured data (CSVs) and unstructured policy documents (Markdown).

## Submission Format

| Column | Type | Description |
|--------|------|-------------|
| `answerable` | 0 or 1 | 1 = answerable, 0 = out-of-scope |
| `lookup_value` | float | Raw data value |
| `rule_value` | float | Policy rule applied (0 if no rule needed) |
| `final_answer` | float | Computed result |

**Out-of-scope:** `answerable=0, lookup_value=-1, rule_value=-1, final_answer=-1`

**Evaluation:** RMSE across all 4 columns. Lower is better.

---

> Follow the cells sequentially. Complete sections marked TODO.
>
> **IMPORTANT:** The submission generation cell (Cell 11) MUST be executed for your notebook to produce a valid submission on Kaggle.

## Cell 1 -- Install Dependencies

> Run this cell first. After installation, restart the kernel and run all cells from the top.

In [ ]:
print("Installing required packages...\n")

!pip install -q \
    openai \
    google-genai \
    faiss-cpu \
    pandas \
    numpy

print("\nInstallation complete.")
print("Please restart the kernel/runtime before running the next cell.")

## Cell 2 -- Configuration

Set your data path and model preferences here. All remaining cells use this configuration.

In [ ]:
# -- Configuration --
DATA_PATH = "/kaggle/input/student-performance-parent-call-engine"

LLM_MODEL = "llama-3.3-70b-versatile"
EMBED_MODEL = "gemini-embedding-001"

print(f"Data path: {DATA_PATH}")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")

## Cell 3 -- Imports

> Run this cell without modification.

In [ ]:
import os, json, re, time, csv, math
import numpy as np
import pandas as pd
from datetime import date
from collections import defaultdict
import faiss
from google import genai
from openai import OpenAI

print("Imports loaded successfully.")

## Cell 4 -- API Keys

Add your API keys as **Kaggle Secrets** (Settings > Secrets) before running this cell:
- `GROQ_API_KEY_1`, `GROQ_API_KEY_2`, ... (multiple recommended)
- `GOOGLE_API_KEY`

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()

    GROQ_KEYS = []
    for i in range(1, 9):
        try:
            key = secrets.get_secret(f"GROQ_API_KEY_{i}")
            if key:
                GROQ_KEYS.append(key)
        except:
            break

    GOOGLE_API_KEY = secrets.get_secret("GOOGLE_API_KEY")
    print(f"Running on Kaggle -- {len(GROQ_KEYS)} Groq keys loaded.")

except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    GROQ_KEYS = [os.getenv("GROQ_API_KEY_1", "")]
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
    print("Running locally -- loaded from .env")

# Initialize clients
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

current_key_index = 0
groq_client = OpenAI(
    api_key=GROQ_KEYS[0],
    base_url="https://api.groq.com/openai/v1"
)

def rotate_key():
    """Round-robin key rotation to spread rate limits."""
    global groq_client, current_key_index
    current_key_index = (current_key_index + 1) % len(GROQ_KEYS)
    groq_client = OpenAI(
        api_key=GROQ_KEYS[current_key_index],
        base_url="https://api.groq.com/openai/v1"
    )

print("Clients initialized.")

## Cell 5 -- Load Data

> This cell is pre-filled -- run it to load the dataset.

In [ ]:
# Load structured data
courses_df = pd.read_csv(f"{DATA_PATH}/Dataset/courses.csv")
mentors_df = pd.read_csv(f"{DATA_PATH}/Dataset/mentors.csv")
students_df = pd.read_csv(f"{DATA_PATH}/Dataset/students.csv")
assessments_df = pd.read_csv(f"{DATA_PATH}/Dataset/assessments.csv")
attendance_df = pd.read_csv(f"{DATA_PATH}/Dataset/attendance.csv")

# Load test queries
test_df = pd.read_csv(f"{DATA_PATH}/Query Files/test.csv")

# Load training data (for validation)
train_df = pd.read_csv(f"{DATA_PATH}/Query Files/train.csv")

# Load policy documents
policy_docs = {}
policy_dir = f"{DATA_PATH}/Policy Documents"
for fname in os.listdir(policy_dir):
    if fname.endswith(".md"):
        with open(os.path.join(policy_dir, fname)) as f:
            policy_docs[fname] = f.read()

print(f"Courses: {len(courses_df)}, Mentors: {len(mentors_df)}, Students: {len(students_df)}")
print(f"Assessments: {len(assessments_df)}, Attendance: {len(attendance_df)}")
print(f"Test queries: {len(test_df)}, Train queries: {len(train_df)}")
print(f"Policy documents: {list(policy_docs.keys())}")

## Cell 6 -- Build Vector Store

### Your Task

Build a FAISS vector store from the policy documents for retrieval.

In [ ]:
# TODO: Chunk policy documents, embed them, and build a FAISS index
policy_chunks = []


# TODO: Implement retrieval function
def retrieve_policies(query, top_k=3):
    """Retrieve the top-k most relevant policy chunks for a query."""
    pass

print(f"Created {len(policy_chunks)} chunks")

## Cell 7 -- LLM Helper

> This cell is pre-filled -- modify parameters if needed.

In [ ]:
def call_llm(messages, tools=None, temperature=0.0, max_retries=5):
    """Call Groq LLM with retry and key rotation."""
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": LLM_MODEL,
                "messages": messages,
                "temperature": temperature,
            }
            if tools:
                kwargs["tools"] = tools
                kwargs["tool_choice"] = "auto"

            response = groq_client.chat.completions.create(**kwargs)
            time.sleep(2)
            return response
        except Exception as e:
            rotate_key()
            wait = min(2 ** attempt * 5, 60)
            print(f"  [retry {attempt+1}/{max_retries}] {e} -- waiting {wait}s...")
            time.sleep(wait)

    raise RuntimeError("LLM failed after all retries")

print("LLM helper ready.")

## Cell 8 -- Define Agent Tools

### Your Task

Define the tool functions your agent will call and their OpenAI-format schemas.

Refer to the policy documents and dataset to determine what tools are needed and how they should compute results.

In [ ]:
# TODO: Implement your tool functions
# Each tool should return a JSON string with the computed result.


# Map tool names to functions
TOOL_FUNCTIONS = {
    # "tool_name": tool_function,
}

# TODO: Define OpenAI-format tool schemas
openai_tools = [
    # {
    #     "type": "function",
    #     "function": {
    #         "name": "...",
    #         "description": "...",
    #         "parameters": { ... }
    #     }
    # },
]

print(f"{len(TOOL_FUNCTIONS)} tools defined, {len(openai_tools)} schemas registered.")

## Cell 9 -- Build Agent Pipeline

### Your Task

Implement the agent that takes a query, retrieves relevant policies, calls tools as needed, and returns the answer as a 4-column JSON.

In [ ]:
# TODO: Define your system prompt
SYSTEM_PROMPT = """
"""

# TODO: Implement the agent loop
def run_agent(query, policy_context, max_turns=6):
    """Run the tool-calling agent. Returns the final response text."""
    pass


# Response parsing
def parse_response(text):
    """Extract the 4-column JSON from the agent's response."""
    patterns = [
        r'```json\s*({.*?})\s*```',
        r'({\s*"answerable"\s*:.*?})',
    ]
    for pat in patterns:
        m = re.search(pat, text, re.DOTALL)
        if m:
            try:
                data = json.loads(m.group(1))
                result = {
                    "answerable": int(data.get("answerable", 0)),
                    "lookup_value": float(data.get("lookup_value", -1)),
                    "rule_value": float(data.get("rule_value", -1)),
                    "final_answer": float(data.get("final_answer", -1)),
                }
                if result["answerable"] == 0:
                    result["lookup_value"] = -1
                    result["rule_value"] = -1
                    result["final_answer"] = -1
                return result
            except (json.JSONDecodeError, ValueError):
                continue

    return {"answerable": 0, "lookup_value": -1, "rule_value": -1, "final_answer": -1}

print("Agent pipeline ready.")

## Cell 10 -- Test Your Agent

Test with training queries to validate your pipeline before running on test data.

In [ ]:
# Pick a few training queries to test
sample_queries = train_df.sample(3, random_state=42)

for _, row in sample_queries.iterrows():
    qid = row["query_id"]
    query = row["query"]
    print(f"\n{'='*60}")
    print(f"[{qid}] {query[:100]}...")
    print(f"EXPECTED: a={row['answerable']}, l={row['lookup_value']}, r={row['rule_value']}, f={row['final_answer']}")

    # TODO: Uncomment once your pipeline is implemented
    # policies = retrieve_policies(query)
    # policy_text = "\n".join([p["text"] for p in policies])
    # response = run_agent(query, policy_text)
    # answer = parse_response(response)
    # print(f"PREDICTED: a={answer['answerable']}, l={answer['lookup_value']}, r={answer['rule_value']}, f={answer['final_answer']}")

print("\nTest complete.")

## Cell 11 -- Generate Submission

Process all 50 test queries and write `submission.csv`.

> **This cell MUST be executed** for your notebook to produce a valid Kaggle submission.
>
> Do not modify the output format.

In [ ]:
results = []
errors = []

print(f"Processing {len(test_df)} test queries...\n")

for i, row in test_df.iterrows():
    qid = row["query_id"]
    query = row["query"]
    rotate_key()

    print(f"[{i+1}/{len(test_df)}] {qid}...", end=" ")

    try:
        # Step 1: Retrieve relevant policies
        policies = retrieve_policies(query, top_k=3)
        policy_text = "\n---\n".join([p["text"] for p in policies])

        # Step 2: Run agent
        response = run_agent(query, policy_text)

        # Step 3: Parse response
        answer = parse_response(response)
        answer["query_id"] = qid
        results.append(answer)

        print(f"a={answer['answerable']}, f={answer['final_answer']}")

    except Exception as e:
        print(f"ERROR: {e}")
        errors.append((qid, str(e)))
        results.append({
            "query_id": qid,
            "answerable": 0,
            "lookup_value": -1,
            "rule_value": -1,
            "final_answer": -1,
        })

print(f"\nDone. {len(results)} results, {len(errors)} errors.")

# Write submission
submission = pd.DataFrame(results)[["query_id", "answerable", "lookup_value", "rule_value", "final_answer"]]
submission.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv written ({len(submission)} rows).")

## Cell 12 -- Final Checklist

> Run this cell to validate your submission before uploading.

In [ ]:
print("Final Submission Check")
print("=" * 50)

if os.path.exists("submission.csv"):
    sub = pd.read_csv("submission.csv")

    checks = {
        "File exists": True,
        "Has 50 rows": len(sub) == 50,
        "Has required columns": all(c in sub.columns for c in ["query_id", "answerable", "lookup_value", "rule_value", "final_answer"]),
        "answerable is 0 or 1": sub["answerable"].isin([0, 1]).all(),
        "No NaN values": not sub.isnull().any().any(),
    }

    for check, passed in checks.items():
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {check}")

    oos_count = (sub["answerable"] == 0).sum()
    print(f"\n  Answerable: {len(sub) - oos_count}, Out-of-scope: {oos_count}")

    if all(checks.values()):
        print("\nAll checks passed. Ready to submit!")
    else:
        print("\nSome checks failed. Fix issues and regenerate.")
else:
    print("submission.csv not found. Run Cell 11 first.")

print("=" * 50)
print("Upload submission.csv to Kaggle to complete your submission.")